# PUC Minas Virtual

## Métodos de Programação Aplicados ao Mercado Financeiro

Este é o 18º notebook da nossa disciplina, sobre um empreendimento imobiliário, considerado como um investimento de renda fixa.

Autor: Prof. Leopoldo Grajeda

Data da última atualização: 21/04/2023

In [1]:
# Carregamento das bibliotecas
import numpy as np
import pandas as pd
import scipy.optimize as optimize    # Pacote de otimização

### Parte I - Problema

#### Empreendimento imobiliário

Uma construtora está considerando a possibilidade de realizar um empreendimento imobiliário, que consiste na construção de um edifício com 60 salas comerciais. 

Conforme um orçamento preliminar, o empreendimento terá os seguintes custos:
* Aquisição do terreno, a ser comprado por R$\$ $ 2.000.000. 
* Projeto arquitetônico no valor de R$\$ $ 500.000, a ser pago de 2 vezes: um adiantamento de 30% um mês após a compra do terreno e o restante na entrega do projeto 6 meses após a compra do terreno.
* Enquanto o projeto arquitetônico é elaborado, a demolição será feita em 4 meses, ao custo de R$\$ $ 10.000 por mês.
* No mês seguinte à entrega do projeto, será feita a incorporação, com gasto estimado em R$\$ $ 100.000, já incluindo uma pequena campanha de marketing.
* No mês seguinte à incorporação, será feito o lançamento do empreendimento, com custo previsto em R$\$ $ 100.000.
* Um mês depois, terá início a obra, com a fundação do edifício sendo realizada por uma empresa especializada, ao custo de R$\$ $ 1.000.000.
* Nos 14 meses subsequentes à fundação, será feita construção do edifício, com despesas previstas em R$\$ $ 200.000 por mês.

Por outro lado, há um faturamento previsto com a venda das unidades ao longo do tempo:
* Durante o lançamento da campanha de marketing, cada sala será comercializada pelo preço promocional de R$\$ $ 120.000, já descontadas as despesas de corretagem. Nessa ocasião, espera-se vender 5 unidades.
* A partir daí, o preço de comercialização será aumentado em R$\$ $ 5.000 a cada mês até a conclusão da obra, com expectativa de venda de 1 unidade por mês, exceto nos últimos 4 meses de obra, quando esse número deve aumentar para 2 unidades mensais.
* Após a conclusão das obras, as salas serão negociadas pelo valor de mercado, previsto para R$ 220.000. Espera-se vender 3 unidades por mês, até que todas as unidades estejam vendidas.

Os fluxos de caixa desse empreendimento foram colocados em uma planilha Excel anexa.

In [2]:
# Carregamento da planilha com os fluxos de caixa
Empreendimento = pd.read_excel('Empreendimento.xlsx')

# Exibe o DataFrame com os dados
Empreendimento

,Mês,Etapa,Despesa,Receita,Unnamed: 4,Unnamed: 5,Unnamed: 6
0,0.0,Terreno,2000000.0,NaN,NaN,NaN,NaN
1,1.0,Projeto 1,150000.0,NaN,NaN,NaN,NaN
2,2.0,Demolição 1,10000.0,NaN,NaN,NaN,NaN
3,3.0,Demolição 2,10000.0,NaN,NaN,NaN,NaN
4,4.0,Demolição 3,10000.0,NaN,NaN,NaN,NaN
5,5.0,Demolição 4,10000.0,NaN,NaN,NaN,NaN
6,6.0,Projeto 2,350000.0,NaN,NaN,FATURAMENTO DETALHADO,NaN
7,7.0,Incorporação,100000.0,NaN,NaN,Preço,Unidades
8,8.0,Lançamento,100000.0,600000.0,NaN,120000,5
9,9.0,Fundação,1000000.0,125000.0,NaN,125000,1


In [3]:
# Tratamento do DataFrame

# Restrição aos fluxos de caixa relevantes, desprezando as linhas e colunas que não interessam
Fluxos = Empreendimento[['Despesa','Receita']][:-2]

# Substitui valores faltantes por zero
Fluxos = Fluxos.replace(to_replace = np.nan,value = 0)

# Ajusta exibição do DataFrame com duas casas decimais
pd.options.display.float_format = "{:,.2f}".format

# Exibe o DataFrame
Fluxos

,Despesa,Receita
0,"2,000,000.00",0.00
1,"150,000.00",0.00
2,"10,000.00",0.00
3,"10,000.00",0.00
4,"10,000.00",0.00
5,"10,000.00",0.00
6,"350,000.00",0.00
7,"100,000.00",0.00
8,"100,000.00","600,000.00"
9,"1,000,000.00","125,000.00"


In [4]:
# Cálculo do vetor de fluxos de caixa líquidos

FluxosLiq = Fluxos['Receita'] - Fluxos['Despesa']   # Consolida o fluxo de caixa líquido em cada mês

### Parte II - Cálculos para o empreendimento

#### Cenário esperado

Para executar os cálculos, é preciso definir a taxa de juros esperada para o período do empreendimento.

Idealmente, deve-se usar o custo de capital (WACC) da construtora.

In [5]:
# Define a taxa de juros de referência

r = 0.02   # Taxa de juros mensal a ser usada (em % a.m.)

#### Cálculo do valor presente líquido - VPL

O VPL é a soma dos fluxos de caixa descontados pela taxa de juros usada como referência

In [6]:
# Uma variável VPL será usada para acumular os valores presentes de cada fluxo de caixa da série
VPL = 0

# Cálculo do VPL
for t in range(len(FluxosLiq)):
    VPL += FluxosLiq[t]/((1+r)**t)   # Soma o valor presente de cada fluxo de caixa da série
    
# Exibição do resultado
print(f'Taxa de juros de referência {100*r:,.2f} % a.m.')
print(f'VPL do empreendimento: R$ {VPL:,.2f}')

Taxa de juros de referência 2.00 % a.m.
VPL do empreendimento: R$ 1,632,184.48


#### Cálculo do tempo de retorno - $payback$

O $payback$ corresponde ao tempo, em meses, necessário para que o investidor tenha de volta tudo o que investiu. Esse tempo é calculado somando os fluxos de caixa de cada mês, até que essa soma seja positiva.

In [7]:
# Cálculo do Payback

# Uma variável auxiliar será usada para acumular a soma dos fluxos de caixa líquidos
# O payback será o justamente o mês em que essa soma se torne positiva
Soma = FluxosLiq[0]   # A soma começa com o primeiro aporte

# Uma variável Payback será usada para contar os meses 
Payback = 0

# Loop para somar os fluxos, enquanto a soma for negativa 
while Soma < 0:
    Payback += 1                 # Passa para o mês seguinte
    Soma += FluxosLiq[Payback]   # Soma o fluxo do mês
    
# Exibição do resultado
print(f'Payback do empreendimento: {Payback} meses')

Payback do empreendimento: 28 meses


#### Cálculo do taxa interna de retorno - TIR

A TIR é a taxa de juros que faz com que o VPL seja nulo.

In [8]:
# Cálculo da TIR

# Definição do VPL como função da taxa de juros de referência
def f (x):
    # Uma variável VPLaux será usada para acumular os valores presentes de cada fluxo de caixa da série
    VPLaux = 0
    
    # Cálculo do VPL
    for t in range(len(FluxosLiq)):
        VPLaux += FluxosLiq[t]/((1+x)**t)   # Soma o valor presente de cada fluxo de caixa da série
    
    return VPLaux

# Cálculo da TIR como a raiz da função VPL
TIR = optimize.root(f,[0.0001,1]).x[0]

# Exibição do resultado
print(f'TIR do empreendimento: {100*TIR:,.2f} % a.m.')

TIR do empreendimento: 3.65 % a.m.
